# Analysis of learning time series 

# AR(p) synthetic series vs TimesFM 2.5

**Goal:** Compare one-step-ahead forecasts from a fitted **AR(p)** model to **TimesFM** on synthetic **AR(p)** data.

**Setting:**
- **Horizon:** $t{+}1$ only.
- **Window:** Total length **$T = 720$**. The first **360** observations are the initial history; test indices are $k = 360,\ldots,719$.
- **DGP:** Gaussian AR(**p**) with **p ∈ {2, 5}**, **$n = 720$**.
- **Monte Carlo:** `N_REPLICATIONS` draws per `p`, with fresh randomness every run (`np.random.default_rng()`).
- **Metric:** MSE and Diebold–Mariano (HAC).

**Interpretation:** AR is re-fit on each expanding history; TimesFM is inference-only (no training).

## 1. Setup

Set `HF_TOKEN` in repo-root `.env` (or export it in your shell) for Hugging Face downloads.

This notebook uses one sequential path only (no parallel workers, no seed control).

In [50]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import statsmodels.api as sm
import timesfm
from dotenv import load_dotenv
from IPython.display import display
from statsmodels.tsa.arima.model import ARIMA

from tqdm.auto import tqdm as auto_tqdm

USE_TQDM_WIDGETS = False
notebook_tqdm = auto_tqdm
try:
    from ipywidgets import IntProgress as _IntProgress  # noqa: F401
    from tqdm.notebook import tqdm as notebook_tqdm
    USE_TQDM_WIDGETS = True
except Exception:
    USE_TQDM_WIDGETS = False

warnings.filterwarnings("ignore", category=UserWarning, module="statsmodels")

HERE = Path.cwd().resolve()
REPO = HERE if (HERE / "src" / "timesfm").is_dir() else HERE.parent
load_dotenv(REPO / ".env")
sys.path.insert(0, str(REPO / "src"))

In [ ]:
N_REPLICATIONS = 5

N = 720
K_FIRST = 360

VERBOSE_PROGRESS = True
PROGRESS_EVERY_TASKS = 10

## 2. Simulate AR(p) data

Simple recursion: $y_t = \sum_{i=1}^p \phi_i y_{t-i} + \varepsilon_t$. Draw **$\phi_i \sim \mathrm{Uniform}(-1,1)$** until the process is **stationary** (companion matrix eigenvalues strictly inside the unit circle).

In [52]:
def _is_stationary_ar(phi: np.ndarray) -> bool:
    """Causal AR(p) iff companion-matrix eigenvalues have modulus < 1."""
    p = len(phi)
    comp = np.zeros((p, p), dtype=np.float64)
    comp[0, :] = phi
    if p > 1:
        comp[1:, :-1] = np.eye(p - 1)
    return bool(np.all(np.abs(np.linalg.eigvals(comp)) < 1.0 - 1e-9))


def sample_stationary_phi(
    p: int,
    rng: np.random.Generator,
    *,
    low: float = -1.0,
    high: float = 1.0,
    max_tries: int = 50_000,
) -> np.ndarray:
    """Draw phi_i ~ Uniform(low, high) until stationary (causal AR)."""
    for _ in range(max_tries):
        phi = rng.uniform(low, high, size=p)
        if _is_stationary_ar(phi):
            return phi
    raise RuntimeError(f"failed to sample stationary AR({p}) phi after {max_tries} tries")


def simulate_ar(
    p: int,
    n: int,
    rng: np.random.Generator,
) -> tuple[np.ndarray, np.ndarray]:
    phi = sample_stationary_phi(p, rng)
    eps = rng.normal(0.0, 1.0, n)
    y = np.zeros(n, dtype=np.float64)
    y[:p] = eps[:p]
    for t in range(p, n):
        ar_sum = 0.0
        for lag in range(1, p + 1):
            ar_sum += phi[lag - 1] * y[t - lag]
        y[t] = ar_sum + eps[t]
    return y, phi

## 3. One-step AR forecast (classical)

Fit **ARIMA(p,0,0)** on `history` and return the one-step forecast. Returns `None` if estimation fails (rare on short windows).

In [53]:
def forecast_ar(history: np.ndarray, p: int) -> float | None:
    try:
        if len(history) <= p:
            return None
        fit = ARIMA(history.astype(np.float64), order=(p, 0, 0)).fit()
        return float(np.asarray(fit.forecast(steps=1)).ravel()[0])
    except Exception:
        raise Exception


def forecast_ar_with_bands(
    history: np.ndarray,
    p: int,
    bands: tuple[int, ...] = (20, 40, 60, 80),
) -> tuple[float, dict[int, tuple[float, float]]] | None:
    """Return 1-step AR mean plus central intervals for requested coverage bands."""
    try:
        if len(history) <= p:
            return None
        fit = ARIMA(history.astype(np.float64), order=(p, 0, 0)).fit()
        fc = fit.get_forecast(steps=1)
        mean = float(np.asarray(fc.predicted_mean).ravel()[0])
        intervals: dict[int, tuple[float, float]] = {}
        for b in bands:
            alpha = 1.0 - (b / 100.0)
            ci = np.asarray(fc.conf_int(alpha=alpha), dtype=np.float64).reshape(-1, 2)
            intervals[b] = (float(ci[0, 0]), float(ci[0, 1]))
        return mean, intervals
    except Exception:
        return None


def diebold_mariano_hac(
    loss_a: np.ndarray,
    loss_b: np.ndarray,
    maxlags: int | None = None,
) -> tuple[float, float]:
    """DM test for equal predictive accuracy: H0 E[loss_a - loss_b] = 0.

    Uses OLS of the differential on a constant with HAC (Newey–West) covariance.
    Returns (t-statistic, two-sided p-value).
    """
    d = np.asarray(loss_a, dtype=np.float64) - np.asarray(loss_b, dtype=np.float64)
    t = len(d)
    if t < 2:
        return float("nan"), float("nan")
    if maxlags is None:
        maxlags = max(1, int(np.floor(4 * (t / 100) ** (2 / 9))))
    res = sm.OLS(d, np.ones(t)).fit(cov_type="HAC", cov_kwds={"maxlags": maxlags})
    return float(res.tvalues[0]), float(res.pvalues[0])

## 4. Load TimesFM (once)

Load TimesFM once for the sequential loop in §5 (`horizon = 1`).

In [54]:
torch.set_float32_matmul_precision("high")
tfm = timesfm.TimesFM_2p5_200M_torch.from_pretrained(
    "google/timesfm-2.5-200m-pytorch",
    torch_compile=False,
)
tfm.compile(
    timesfm.ForecastConfig(
        max_context=1024,
        max_horizon=256,
        normalize_inputs=True,
        use_continuous_quantile_head=True,
        force_flip_invariance=True,
        infer_is_positive=False,
        fix_quantile_crossing=True,
    )
)

## 5. Monte Carlo: MSE, DM, and interval diagnostics

For **`N_REPLICATIONS`** draws, simulate a fresh AR series with unseeded randomness and evaluate one-step forecasts on $k = 360,\ldots,719$.

Metrics computed per replication:
- **Point forecast:** MSE and Diebold-Mariano (HAC).
- **Uncertainty bands:** central **20/40/60/80%** interval **coverage** and **average width** for both AR and TimesFM.

In [55]:
rows: list[dict] = []
test_ks = list(range(K_FIRST, N))

BANDS = (20, 40, 60, 80)
# Quantile array layout from TimesFM: [mean, q10, q20, ..., q90].
TF_IDX = {
    20: (4, 6),
    40: (3, 7),
    60: (2, 8),
    80: (1, 9),
}

total_tasks = N_REPLICATIONS
completed_tasks = 0
p = 5
for rep in range(N_REPLICATIONS):
    rng = np.random.default_rng()
    y, phi = simulate_ar(p, N, rng)
    se_ar: list[float] = []
    se_tf: list[float] = []
    cov_ar: dict[int, list[float]] = {b: [] for b in BANDS}
    cov_tf: dict[int, list[float]] = {b: [] for b in BANDS}
    wid_ar: dict[int, list[float]] = {b: [] for b in BANDS}
    wid_tf: dict[int, list[float]] = {b: [] for b in BANDS}
    n_fail = 0

    if USE_TQDM_WIDGETS:
        try:
            tf_bar = notebook_tqdm(
                total=len(test_ks),
                desc=f"rep={rep} p={p} TimesFM",
                position=0,
                leave=True,
                dynamic_ncols=True,
                display=False,
            )
            ar_bar = notebook_tqdm(
                total=len(test_ks),
                desc=f"rep={rep} p={p} AR({p})",
                position=1,
                leave=True,
                dynamic_ncols=True,
                display=False,
            )
            display(tf_bar.container)
            display(ar_bar.container)
        except Exception:
            tf_bar = auto_tqdm(
                total=len(test_ks),
                desc=f"rep={rep} p={p} TimesFM",
                leave=True,
                dynamic_ncols=True,
            )
            ar_bar = auto_tqdm(
                total=len(test_ks),
                desc=f"rep={rep} p={p} AR({p})",
                leave=True,
                dynamic_ncols=True,
            )
    else:
        tf_bar = auto_tqdm(
            total=len(test_ks),
            desc=f"rep={rep} p={p} TimesFM",
            leave=True,
            dynamic_ncols=True,
        )
        ar_bar = auto_tqdm(
            total=len(test_ks),
            desc=f"rep={rep} p={p} AR({p})",
            leave=True,
            dynamic_ncols=True,
        )

    try:
        for k in test_ks:
            actual = float(y[k])
            hist = y[:k].astype(np.float32)

            ar_out = forecast_ar_with_bands(y[:k], p, bands=BANDS)
            ar_bar.update(1)

            tf_point, tf_quantiles = tfm.forecast(horizon=1, inputs=[hist])
            pred_tf = float(tf_point[0, 0])
            tf_q = tf_quantiles[0, 0]
            tf_bar.update(1)

            if ar_out is None:
                n_fail += 1
                continue

            pred_ar, ar_intervals = ar_out

            e_ar = actual - pred_ar
            e_tf = actual - pred_tf
            se_ar.append(e_ar**2)
            se_tf.append(e_tf**2)

            for b in BANDS:
                ar_lo, ar_hi = ar_intervals[b]
                tf_lo, tf_hi = float(tf_q[TF_IDX[b][0]]), float(tf_q[TF_IDX[b][1]])

                cov_ar[b].append(float(ar_lo <= actual <= ar_hi))
                cov_tf[b].append(float(tf_lo <= actual <= tf_hi))
                wid_ar[b].append(ar_hi - ar_lo)
                wid_tf[b].append(tf_hi - tf_lo)
    finally:
        tf_bar.close()
        ar_bar.close()

    se_ar_a = np.asarray(se_ar, dtype=np.float64)
    se_tf_a = np.asarray(se_tf, dtype=np.float64)
    dm_t, dm_p = diebold_mariano_hac(se_ar_a, se_tf_a)

    row = {
        "rep": rep,
        "p_dgp": p,
        "phi": ",".join(f"{x:.8f}" for x in phi),
        "n": N,
        "k_first": K_FIRST,
        "n_test": len(test_ks),
        "ar_fit_failures": n_fail,
        "mse_ar": float(np.mean(se_ar_a)) if len(se_ar_a) else float("nan"),
        "mse_timesfm": float(np.mean(se_tf_a)) if len(se_tf_a) else float("nan"),
        "dm_t_stat": dm_t,
        "dm_pvalue_two_sided": dm_p,
    }

    for b in BANDS:
        row[f"coverage{b}_ar"] = float(np.mean(cov_ar[b])) if cov_ar[b] else float("nan")
        row[f"coverage{b}_timesfm"] = float(np.mean(cov_tf[b])) if cov_tf[b] else float("nan")
        row[f"width{b}_ar"] = float(np.mean(wid_ar[b])) if wid_ar[b] else float("nan")
        row[f"width{b}_timesfm"] = float(np.mean(wid_tf[b])) if wid_tf[b] else float("nan")

    rows.append(row)
    completed_tasks += 1
    if VERBOSE_PROGRESS and (
        completed_tasks % max(1, PROGRESS_EVERY_TASKS) == 0 or completed_tasks == total_tasks
    ):
        print(f"completed {completed_tasks}/{total_tasks} tasks", flush=True)

results = pd.DataFrame(rows)

summary_base = results.groupby("p_dgp", as_index=False).agg(
    n=("rep", "count"),
    ar_fit_failures_total=("ar_fit_failures", "sum"),
    dm_pvalue_median=("dm_pvalue_two_sided", "median"),
)

mse_table = summary_base.merge(
    results.groupby("p_dgp", as_index=False).agg(
        mse_ar_mean=("mse_ar", "mean"),
        mse_ar_std=("mse_ar", "std"),
        mse_timesfm_mean=("mse_timesfm", "mean"),
        mse_timesfm_std=("mse_timesfm", "std"),
    ),
    on="p_dgp",
)
mse_table["mse_diff_timesfm_minus_ar"] = (
    mse_table["mse_timesfm_mean"] - mse_table["mse_ar_mean"]
)
mse_table["better_mse_model"] = np.where(
    mse_table["mse_diff_timesfm_minus_ar"] < 0, "TimesFM", "AR"
)

coverage_records: list[dict[str, float | int | str]] = []
width_records: list[dict[str, float | int | str]] = []
for p_val, g in results.groupby("p_dgp"):
    for b in BANDS:
        target = b / 100.0
        cov_ar_mean = float(g[f"coverage{b}_ar"].mean())
        cov_tfm_mean = float(g[f"coverage{b}_timesfm"].mean())
        wid_ar_mean = float(g[f"width{b}_ar"].mean())
        wid_tfm_mean = float(g[f"width{b}_timesfm"].mean())

        coverage_records.append(
            {
                "p_dgp": int(p_val),
                "band": f"{b}%",
                "target_coverage": target,
                "coverage_ar": cov_ar_mean,
                "coverage_timesfm": cov_tfm_mean,
                "abs_calib_error_ar": abs(cov_ar_mean - target),
                "abs_calib_error_timesfm": abs(cov_tfm_mean - target),
                "better_calibration": "AR"
                if abs(cov_ar_mean - target) < abs(cov_tfm_mean - target)
                else "TimesFM",
            }
        )
        width_records.append(
            {
                "p_dgp": int(p_val),
                "band": f"{b}%",
                "width_ar": wid_ar_mean,
                "width_timesfm": wid_tfm_mean,
                "width_diff_timesfm_minus_ar": wid_tfm_mean - wid_ar_mean,
                "narrower_model": "TimesFM" if wid_tfm_mean < wid_ar_mean else "AR",
            }
        )

coverage_table = pd.DataFrame(coverage_records)
width_table = pd.DataFrame(width_records)

coverage_long = coverage_table.melt(
    id_vars=["p_dgp", "band", "target_coverage"],
    value_vars=["coverage_ar", "coverage_timesfm"],
    var_name="model_metric",
    value_name="coverage",
)
coverage_long["model"] = coverage_long["model_metric"].str.replace("coverage_", "", regex=False)
coverage_long["abs_calibration_error"] = (
    coverage_long["coverage"] - coverage_long["target_coverage"]
).abs()

width_long = width_table.melt(
    id_vars=["p_dgp", "band"],
    value_vars=["width_ar", "width_timesfm"],
    var_name="model_metric",
    value_name="avg_width",
)
width_long["model"] = width_long["model_metric"].str.replace("width_", "", regex=False)

detailed_summary = coverage_long.merge(
    width_long[["p_dgp", "band", "model", "avg_width"]],
    on=["p_dgp", "band", "model"],
    how="left",
).sort_values(["p_dgp", "band", "model"]).reset_index(drop=True)

# Backward-compatible combined summary table retained for CSV export.
summary = mse_table.merge(
    coverage_table.pivot(index="p_dgp", columns="band", values="coverage_ar")
    .rename(columns=lambda c: f"coverage{c}_ar")
    .reset_index(),
    on="p_dgp",
    how="left",
).merge(
    coverage_table.pivot(index="p_dgp", columns="band", values="coverage_timesfm")
    .rename(columns=lambda c: f"coverage{c}_timesfm")
    .reset_index(),
    on="p_dgp",
    how="left",
).merge(
    width_table.pivot(index="p_dgp", columns="band", values="width_ar")
    .rename(columns=lambda c: f"width{c}_ar")
    .reset_index(),
    on="p_dgp",
    how="left",
).merge(
    width_table.pivot(index="p_dgp", columns="band", values="width_timesfm")
    .rename(columns=lambda c: f"width{c}_timesfm")
    .reset_index(),
    on="p_dgp",
    how="left",
)

print("MSE / DM Summary")
display(mse_table)
print("Coverage Summary")
display(coverage_table)
print("Width Summary")
display(width_table)
print("Detailed Summary (long format)")
display(detailed_summary)
print("Per-replication rows (preview)")
display(results.head(8))

rep=0 p=5 TimesFM:   0%|          | 0/360 [00:00<?, ?it/s]

rep=0 p=5 AR(5):   0%|          | 0/360 [00:00<?, ?it/s]

rep=1 p=5 TimesFM:   0%|          | 0/360 [00:00<?, ?it/s]

rep=1 p=5 AR(5):   0%|          | 0/360 [00:00<?, ?it/s]

rep=2 p=5 TimesFM:   0%|          | 0/360 [00:00<?, ?it/s]

...

,p_dgp,band,width_ar,width_timesfm,width_diff_timesfm_minus_ar,narrower_model
0,5,20%,0.508036,0.547934,0.039898,AR
1,5,40%,1.051578,1.107003,0.055425,AR
2,5,60%,1.687699,1.814511,0.126812,AR
3,5,80%,2.569889,2.844799,0.274910,AR


Detailed Summary (long format)


,p_dgp,band,target_coverage,model_metric,coverage,model,abs_calibration_error,avg_width
0,5,20%,0.2,coverage_ar,0.208889,ar,0.008889,0.508036
1,5,20%,0.2,coverage_timesfm,0.212222,timesfm,0.012222,0.547934
2,5,40%,0.4,coverage_ar,0.387222,ar,0.012778,1.051578
3,5,40%,0.4,coverage_timesfm,0.408889,timesfm,0.008889,1.107003
4,5,60%,0.6,coverage_ar,0.605000,ar,0.005000,1.687699
5,5,60%,0.6,coverage_timesfm,0.611667,timesfm,0.011667,1.814511
6,5,80%,0.8,coverage_ar,0.799444,ar,0.000556,2.569889
7,5,80%,0.8,coverage_timesfm,0.822222,timesfm,0.022222,2.844799


Per-replication rows (preview)


,rep,p_dgp,phi,n,k_first,n_test,ar_fit_failures,mse_ar,mse_timesfm,dm_t_stat,...,width40_ar,width40_timesfm,coverage60_ar,coverage60_timesfm,width60_ar,width60_timesfm,coverage80_ar,coverage80_timesfm,width80_ar,width80_timesfm
0,0,5,"0.11199010,-0.70500013,0.35021643,-0.45821379,...",720,360,360,0,0.995846,1.057827,-2.295305,...,1.071615,1.126908,0.636111,0.647222,1.719856,1.844306,0.833333,0.833333,2.618856,2.872093
1,1,5,"-0.28307559,0.14198047,0.43789649,-0.16247471,...",720,360,360,0,0.912821,0.948244,-2.171964,...,1.016169,1.021162,0.597222,0.588889,1.630870,1.676210,0.783333,0.791667,2.483355,2.604768
2,2,5,"-0.55777209,-0.24162473,0.36898875,0.36431822,...",720,360,360,0,1.116067,1.235368,-2.760591,...,1.118871,1.257602,0.627778,0.655556,1.795699,2.053232,0.797222,0.852778,2.734343,3.204524
3,3,5,"0.49251271,0.10244153,0.51570115,0.30631215,-0...",720,360,360,0,0.879868,0.966678,-3.003747,...,1.000733,1.082334,0.575000,0.605556,1.606097,1.772912,0.791667,0.825000,2.445633,2.808854
4,4,5,"0.80356554,-0.23523655,0.20658389,0.08221548,-...",720,360,360,0,1.050275,1.091856,-2.194033,...,1.050501,1.047010,0.588889,0.561111,1.685971,1.725898,0.791667,0.808333,2.567258,2.733756


## 6. Save tables

Save both raw outputs and readable summary tables under `output/` next to this notebook:
- per-replication rows,
- combined summary,
- MSE/DM summary,
- coverage-by-band summary,
- width-by-band summary,
- detailed long-format summary.

In [56]:
out_dir = HERE / "output"
out_dir.mkdir(parents=True, exist_ok=True)

path_rep = out_dir / "ar_vs_timesfm_replications.csv"
path_sum = out_dir / "ar_vs_timesfm_summary.csv"
path_mse = out_dir / "ar_vs_timesfm_mse_summary.csv"
path_cov = out_dir / "ar_vs_timesfm_coverage_summary.csv"
path_wid = out_dir / "ar_vs_timesfm_width_summary.csv"
path_detail = out_dir / "ar_vs_timesfm_detailed_summary.csv"

results.to_csv(path_rep, index=False)
summary.to_csv(path_sum, index=False)
mse_table.to_csv(path_mse, index=False)
coverage_table.to_csv(path_cov, index=False)
width_table.to_csv(path_wid, index=False)
detailed_summary.to_csv(path_detail, index=False)

print(path_rep)
print(path_sum)
print(path_mse)
print(path_cov)
print(path_wid)
print(path_detail)

/Users/nikhileshbelulkar/Documents/timesfm_google/experiments_nikhilesh/output/ar_vs_timesfm_replications.csv
/Users/nikhileshbelulkar/Documents/timesfm_google/experiments_nikhilesh/output/ar_vs_timesfm_summary.csv
/Users/nikhileshbelulkar/Documents/timesfm_google/experiments_nikhilesh/output/ar_vs_timesfm_mse_summary.csv
/Users/nikhileshbelulkar/Documents/timesfm_google/experiments_nikhilesh/output/ar_vs_timesfm_coverage_summary.csv
/Users/nikhileshbelulkar/Documents/timesfm_google/experiments_nikhilesh/output/ar_vs_timesfm_width_summary.csv
/Users/nikhileshbelulkar/Documents/timesfm_google/experiments_nikhilesh/output/ar_vs_timesfm_detailed_summary.csv
